In [ ]:
# ═══════════════════════════════════════════════════════════════
# Sensitivity of Issued Credits for Smallholder Agroforestry
# under a VM0047 (ARR) Framework — India CCTS context undefined
# Reproducible open-data pipeline. Arecanut agroforestry, Karnataka.
# Methodology basis: VM0047 v1.1 (Verra ARR), successor to CDM AR-ACM0003.
# ═══════════════════════════════════════════════════════════════

# ── Cell 1: Environment ──
!pip install earthengine-api geemap numpy pandas matplotlib -q
import ee, geemap, numpy as np, pandas as pd, matplotlib.pyplot as plt
import itertools, json, hashlib
ee.Authenticate()
ee.Initialize(project='YOUR-EE-PROJECT-ID')

In [ ]:
# ── Cell 2: Study parcel ──
# Arecanut agroforestry parcel, Chikkamagaluru (Malnad), Karnataka, India.
# Species composition assumed from regional cropping patterns; not derivable
# from 10 m optical imagery. Established by field inventory in deployment.
coords = [
    [75.7583337, 13.4004903], [75.7581502, 13.3995258],
    [75.7593232, 13.3995341], [75.7594960, 13.4003524],
    [75.7583337, 13.4004903],
]
aoi = ee.Geometry.Polygon([coords])
AREA_HA = aoi.area().getInfo() / 10000
print(f"Parcel area: {AREA_HA:.2f} ha")

In [ ]:
# ── Cell 3: Sentinel-2 composite, NDVI, and map figure ──
# Dry-season cloud-filtered median composite. NDVI confirms dense live canopy;
# it is not used as a biomass predictor (biomass is allometric).
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi).filterDate('2024-11-01', '2025-03-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .median().clip(aoi))
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
mean_ndvi = ndvi.reduceRegion(ee.Reducer.mean(), aoi, 10).get('NDVI').getInfo()
print(f"Mean NDVI: {mean_ndvi:.3f}")

m = geemap.Map()
m.centerObject(aoi, 16)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'True colour')
m.addLayer(ndvi, {'min': 0, 'max': 0.9,
                  'palette': ['white', 'khaki', 'green', 'darkgreen']}, 'NDVI')
m.addLayer(aoi, {'color': 'red'}, 'Parcel boundary')
m

In [ ]:
# ── Cell 4: Methodology parameters and alternatives (all cited) ──
# METHODOLOGY: VM0047 v1.1 (Verra ARR), successor to CDM AR-ACM0003.
#   Pools: above-ground + below-ground biomass; SOC OPTIONAL (excluded here,
#     per VM0047 — biomass restricted to live woody biomass when SOC excluded).
#   Uncertainty: minimum 10% deduction (90% CI); higher permitted. [VM0047 v1.1]
#   Permanence: buffer pool via VCS AFOLU Non-Permanence Risk Tool (risk-rated).
#
# Above-ground biomass, arecanut, combined allometric form:
#   AGB(kg) = 0.03883 * H * D^1.2  (R^2=0.96)
#   Prayogo et al. (2018), AGRIVITA 40(3):381-389, doi:10.17503/agrivita.v40i3.1124
# Carbon fraction: 0.47 (IPCC 2006 Vol.4) / 0.50 (palm-carbon literature)
# Root-to-shoot ratio: 0.15 (Pragasan 2025, arecanut) / 0.24 (IPCC 2006 tropical)
# Uncertainty deduction: 10% (VM0047 minimum) / 15% / 20% (higher, permitted)
# Permanence buffer: 10% / 20% / 40% (VCS AFOLU NPRT risk-rated range)
PARAMS = {
    'agb': {'k': 0.03883, 'd_exp': 1.2},
    'carbon_fraction':   [0.47, 0.50],
    'rsr':               [0.15, 0.24],
    'uncertainty_ded':   [0.10, 0.15, 0.20],   # VM0047 minimum is 10%
    'permanence_buffer': [0.10, 0.20, 0.40],   # VCS AFOLU NPRT
    'co2e': 44 / 12,
}

In [ ]:
# ── Cell 5: Carbon accounting functions (VM0047 woody-biomass basis) ──
def areca_agb_kg(D_cm, H_m, k=0.03883, d_exp=1.2):
    """Above-ground biomass per palm (kg); Prayogo et al. (2018) combined form."""
    return k * H_m * (D_cm ** d_exp)

def agb_increment_tC_ha(D0, H0, D1, H1, n_per_ha, carbon_fraction):
    """Annual above-ground carbon increment (tC/ha/yr). Credits change, not stock."""
    a0 = areca_agb_kg(D0, H0) * n_per_ha
    a1 = areca_agb_kg(D1, H1) * n_per_ha
    return (a1 - a0) / 1000 * carbon_fraction

def issued_credits(agb_inc_tC_ha, rsr, area_ha, uncertainty_ded,
                   permanence_buffer, co2e=44/12):
    """Net issued credits (tCO2e/yr) under VM0047 woody-biomass accounting:
       (AGB + BGB) - uncertainty deduction - permanence buffer. SOC excluded."""
    bgb = agb_inc_tC_ha * rsr
    gross = (agb_inc_tC_ha + bgb) * co2e * area_ha
    return gross * (1 - uncertainty_ded) * (1 - permanence_buffer)

In [ ]:
# ── Cell 6: Stand parameters (field-measured in deployment) ──
# Documented assumptions for arecanut; held fixed across the sensitivity,
# which varies methodology choices only.
N_PER_HA = 1370          # ~2.7 m x 2.7 m arecanut spacing
D0, D1   = 12.0, 12.0    # stem diameter (cm), ~constant for palms
H0, H1   = 10.0, 10.4    # height (m), year 0 to year 1

In [ ]:
# ── Cell 7: Baseline estimate (one defensible parameter set) ──
base_agb = agb_increment_tC_ha(D0, H0, D1, H1, N_PER_HA, 0.47)
base = issued_credits(base_agb, rsr=0.24, area_ha=AREA_HA,
                      uncertainty_ded=0.10, permanence_buffer=0.20)
print(f"Baseline issued credits: {base:.3f} tCO2e/yr "
      f"(CF=0.47, RSR=0.24, unc=10%, buffer=20%)")

In [ ]:
# ── Cell 8: Sensitivity across defensible methodology choices ──
grid = list(itertools.product(
    PARAMS['carbon_fraction'], PARAMS['rsr'],
    PARAMS['uncertainty_ded'], PARAMS['permanence_buffer']))
rows = []
for cf, rsr, unc, buf in grid:
    a = agb_increment_tC_ha(D0, H0, D1, H1, N_PER_HA, cf)
    iss = issued_credits(a, rsr=rsr, area_ha=AREA_HA,
                         uncertainty_ded=unc, permanence_buffer=buf)
    rows.append({'carbon_fraction': cf, 'rsr': rsr,
                 'uncertainty_ded': unc, 'permanence_buffer': buf, 'issued': iss})
sens = pd.DataFrame(rows)
lo, hi = sens['issued'].min(), sens['issued'].max()
cv = sens['issued'].std() / sens['issued'].mean()
print(f"Combinations evaluated: {len(grid)}")
print(f"Issued credits range: {lo:.3f} - {hi:.3f} tCO2e/yr")
print(f"Spread factor: {hi/lo:.2f}x   CV: {cv:.1%}")

In [ ]:
# ── Cell 9: Parameter contribution (tornado, one-at-a-time) ──
def run(cf, rsr, unc, buf):
    a = agb_increment_tC_ha(D0, H0, D1, H1, N_PER_HA, cf)
    return issued_credits(a, rsr=rsr, area_ha=AREA_HA,
                          uncertainty_ded=unc, permanence_buffer=buf)
b = dict(cf=0.47, rsr=0.24, unc=0.10, buf=0.20)
bv = run(**b)
axes = {
 'Carbon fraction':   [run(v,b['rsr'],b['unc'],b['buf']) for v in [0.47,0.50]],
 'Root-to-shoot':     [run(b['cf'],v,b['unc'],b['buf']) for v in [0.15,0.24]],
 'Uncertainty ded.':  [run(b['cf'],b['rsr'],v,b['buf']) for v in [0.10,0.20]],
 'Permanence buffer': [run(b['cf'],b['rsr'],b['unc'],v) for v in [0.10,0.40]],
}
fig, ax = plt.subplots(figsize=(8,4.5))
for name,(v1,v2) in axes.items():
    ax.barh(name, abs(v2-v1), left=min(v1,v2), color='#2f6db0', alpha=0.85)
ax.axvline(bv, color='k', ls='--', lw=1, label=f'baseline = {bv:.2f}')
ax.set_xlabel('Issued credits (tCO$_2$e yr$^{-1}$)')
ax.set_title('Sensitivity of issued credits to methodology choices (VM0047 ARR)\n'
             'Arecanut agroforestry, 1.27 ha')
ax.legend(); plt.tight_layout()
plt.savefig('figure_sensitivity.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# ── Cell 10: Methodology context — VM0047 vs India CCTS (qualitative) ──
context = pd.DataFrame([
    ['Status', 'VM0047 v1.1 active (May 2025); replaced AR-ACM0003',
     'Offset Procedure Mar 2025; agroforestry a Phase-1 sector, methodology not yet approved'],
    ['Pools', 'AGB + BGB; deadwood/litter/SOC optional',
     'CDM-derived forestry tools cover AGB, BGB, SOC (BM-T-AR-0004/0006)'],
    ['Uncertainty', 'Minimum 10% deduction (90% CI)',
     'Conservativeness principle; no numeric deduction published for agroforestry'],
    ['Permanence', 'Buffer pool via VCS AFOLU NPRT',
     'Not yet specified'],
    ['Verifier', 'VVB', 'ACVA'],
], columns=['Aspect', 'VM0047 (ARR)', 'India CCTS (offset)'])
print(context.to_string(index=False))

In [ ]:
# ── Cell 11: Integrity ledger ──
def hash_chain(records):
    prev = "0"*64; out = []
    for r in records:
        h = hashlib.sha256((json.dumps(r, sort_keys=True)+prev).encode()).hexdigest()
        out.append({**r, 'hash': h}); prev = h
    return out
for r in hash_chain([{'record':'ndvi','value':round(mean_ndvi,3)},
                     {'record':'issued_min','value':round(lo,3)},
                     {'record':'issued_max','value':round(hi,3)}]):
    print(f"{r['record']:<12} {r['hash'][:16]}...")

In [ ]:
# ── Cell 12: Summary ──
print("Sensitivity analysis under VM0047 (ARR); India CCTS context undefined")
print(f"Parcel: {AREA_HA:.2f} ha | NDVI: {mean_ndvi:.3f} | SOC excluded (optional in VM0047)")
print(f"Combinations: {len(grid)} | Issued: {lo:.2f}-{hi:.2f} tCO2e/yr | spread {hi/lo:.2f}x")
print("India CCTS: agroforestry methodology not yet approved; numeric rules undefined")